## Mount Drive and locate project. Verify lightweight import. Install only the offline ASR dependencies.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/aic26/asr_model_comparison")
PRODUCTION_ROOT = PROJECT_ROOT / "production"

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert (PRODUCTION_ROOT / "asr_retrieval").exists()

os.chdir(PROJECT_ROOT)

if (str(PRODUCTION_ROOT) not in sys.path) :
    sys.path.insert(0, str(PRODUCTION_ROOT))

print("Project:", PROJECT_ROOT)
print("Production:", PRODUCTION_ROOT)
print("Python:", sys.version)

PACKAGE_ROOT = PRODUCTION_ROOT / "asr_retrieval"
for path in sorted(PACKAGE_ROOT.glob("*.py")) :
    print(path.name)

import sys
assert "nemo" not in sys.modules

import asr_retrieval

print("Imported:", asr_retrieval)
print("NeMo imported:", "nemo" in sys.modules)

assert "nemo" not in sys.modules

import subprocess
import sys

packages = [
    "Cython",
    "packaging",
    "nemo_toolkit[asr]==2.7.3",
    "huggingface_hub",
    "soundfile",
    "scipy",
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    *packages,
]

print("Installing Phase 1 offline dependencies...")
subprocess.run(command, check = True)

Mounted at /content/drive
Project: /content/drive/MyDrive/aic26/asr_model_comparison
Production: /content/drive/MyDrive/aic26/asr_model_comparison/production
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
__init__.py
audio.py
config.py
postprocess.py
schemas.py
transcriber.py
Imported: <module 'asr_retrieval' from '/content/drive/MyDrive/aic26/asr_model_comparison/production/asr_retrieval/__init__.py'>
NeMo imported: False
Installing Phase 1 offline dependencies...


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'Cython', 'packaging', 'nemo_toolkit[asr]==2.7.3', 'huggingface_hub', 'soundfile', 'scipy'], returncode=0)

## Run deterministic Phase 1 tests first

In [2]:
!python -m pytest tests/production/test_phase1.py -q

........                                                                 [100%]
8 passed in 3.66s


In [3]:
!ffmpeg -version | head -n 1
!ffprobe -version | head -n 1

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
ffprobe version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2007-2021 the FFmpeg developers


In [4]:
import nemo
import torch

print("NeMo:", nemo.__version__)
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

NeMo: 2.7.3
Torch: 2.11.0+cu128
GPU: Tesla T4


## Load configuration

In [1]:
import sys
import importlib.metadata

print("Python:", sys.version)
print("NumPy package:", importlib.metadata.version("numpy"))
print("NeMo package:", importlib.metadata.version("nemo_toolkit"))

import numpy as np

print("NumPy import OK:", np.__version__)
print("NumPy path:", np.__file__)

import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

import nemo
import nemo.collections.asr as nemo_asr

print("NeMo:", nemo.__version__)
print("NeMo ASR import OK")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy package: 2.5.2
NeMo package: 2.7.3
NumPy import OK: 2.5.2
NumPy path: /usr/local/lib/python3.12/dist-packages/numpy/__init__.py
Torch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


[NeMo W 2026-08-12 08:06:09 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.


AttributeError: module 'numba.cuda.types' has no attribute 'NPDatetime'

In [2]:
import importlib.metadata

for package in [
    "numpy",
    "numba",
    "numba-cuda",
    "llvmlite",
    "nemo_toolkit",
] :
    try :
        print(package, importlib.metadata.version(package))
    except Exception as error :
        print(package, "NOT FOUND", error)

numpy 2.5.2
numba 0.67.0
numba-cuda 0.22.2
llvmlite 0.49.0
nemo_toolkit 2.7.3


In [3]:
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--force-reinstall",
    "--no-cache-dir",
    "numpy==2.4.3",
    "numba==0.65.1",
    "llvmlite==0.47.0",
    "numba-cuda==0.30.2",
]

print("Running:", " ".join(command))
subprocess.run(command, check = True)

Running: /usr/bin/python3 -m pip install --force-reinstall --no-cache-dir numpy==2.4.3 numba==0.65.1 llvmlite==0.47.0 numba-cuda==0.30.2


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--force-reinstall', '--no-cache-dir', 'numpy==2.4.3', 'numba==0.65.1', 'llvmlite==0.47.0', 'numba-cuda==0.30.2'], returncode=0)

In [1]:
import sys
import importlib.metadata

packages = [
    "numpy",
    "numba",
    "numba-cuda",
    "llvmlite",
    "nemo_toolkit",
]

print("Python:", sys.version)

for package in packages :
    print(
        f"{package:15s}",
        importlib.metadata.version(package),
    )

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
numpy           2.4.3
numba           0.65.1
numba-cuda      0.30.2
llvmlite        0.47.0
nemo_toolkit    2.7.3


In [2]:
import numpy as np
import numba
import llvmlite

print("NumPy:", np.__version__)
print("Numba:", numba.__version__)
print("llvmlite:", llvmlite.__version__)

from numba import cuda

print("numba.cuda path:")
print(cuda.__file__)

NumPy: 2.4.3
Numba: 0.65.1
llvmlite: 0.47.0
numba.cuda path:
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/__init__.py


In [3]:
import nemo
import nemo.collections.asr as nemo_asr

print("NeMo:", nemo.__version__)
print("NeMo ASR import OK")

[NeMo W 2026-08-12 08:09:25 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
[NeMo W 2026-08-12 08:09:26 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-08-12 08:09:26 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-08-12 08:09:26 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-08-12 08:09:26 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    


NeMo: 2.7.3
NeMo ASR import OK


In [4]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/aic26/asr_model_comparison")
PRODUCTION_ROOT = PROJECT_ROOT / "production"

os.chdir(PROJECT_ROOT)

if (str(PRODUCTION_ROOT) not in sys.path) :
    sys.path.insert(0, str(PRODUCTION_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from asr_retrieval.config import (
    AudioWindowConfig,
    OfflineASRConfig,
    ParakeetConfig,
)

config = OfflineASRConfig(
    audio = AudioWindowConfig(),
    parakeet = ParakeetConfig(
        device = "cuda",
    ),
)

print(config)

print("Sample rate:", config.audio.sample_rate)
print("Window samples:", config.audio.window_samples)
print("Stride samples:", config.audio.stride_samples)
print("Minimum tail:", config.audio.minimum_tail_samples)

assert config.audio.sample_rate == 16_000
assert config.audio.window_samples == 960_000
assert config.audio.stride_samples == 720_000
assert config.audio.minimum_tail_samples == 240_000

from asr_retrieval.transcriber import ParakeetTranscriber

transcriber = ParakeetTranscriber(config.parakeet)

print("Loading Parakeet...")
transcriber.load()

print("Loaded:", transcriber.loaded)
print("Revision:", transcriber.resolved_revision)

OfflineASRConfig(schema_version='1.0', audio=AudioWindowConfig(sample_rate=16000, channels=1, sample_width_bytes=2, window_samples=960000, stride_samples=720000, minimum_tail_samples=240000), parakeet=ParakeetConfig(model_name='nvidia/parakeet-ctc-0.6b-Vietnamese', model_filename='parakeet-ctc-0.6b-vi.nemo', revision='b0493142b49458810324e3db8be9e8e07b4ebc17', device='cuda', timestamps=True, decoder='greedy_ctc', dtype='float32', batch_size=1, model_cache_dir=None))
Sample rate: 16000
Window samples: 960000
Stride samples: 720000
Minimum tail: 240000
Loading Parakeet...


parakeet-ctc-0.6b-vi.nemo:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

[NeMo I 2026-08-12 08:10:31 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-08-12 08:10:31 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - - /datasets/vlsp2020-vinai/tar_files/vlsp2020_tarred_audio_manifest_multitexts_v2_qwen_norm.json
    - - /datasets/VietMed/labeled_medical_data/tar_files/VietMed_tarred_audio_manifest_multitexts_v2_qwen_norm.json
    - - /datasets/FOSD/tar_files/FOSD_tarred_audio_manifest_fixed_wav_multitexts_v2_qwen_norm.json
    - - /datasets/LSVSC/tar_files/LSVSC_tarred_audio_manifest_multitexts_v2_qwen_norm.json
    - - /datasets/VSV-1100/tar_files/VSV1100_tarred_audio_manifest_multitexts_v2_qwen_norm.json
    - - /datasets/cv-corpus-20.0-2024-12-06/vi/tar_files/CV_tarred_audio_manifest_fixed_wav_multitexts_v2_qwen_norm.json
    - - /datasets/fleurs/data/vi_vn/tar_files/fleurs_tarred_audio_manifest_multitexts_v2_qwen_norm.json
    - - /datasets

[NeMo I 2026-08-12 08:10:42 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /root/.cache/huggingface/hub/models--nvidia--parakeet-ctc-0.6b-Vietnamese/snapshots/b0493142b49458810324e3db8be9e8e07b4ebc17/parakeet-ctc-0.6b-vi.nemo.
Loaded: True
Revision: b0493142b49458810324e3db8be9e8e07b4ebc17


### Automatically locate one known canonical benchmark WAV

In [6]:
import json

BENCHMARK_PATH = PROJECT_ROOT / "data" / "benchmark_manifest.json"

benchmark = json.loads(
    BENCHMARK_PATH.read_text(encoding = "utf-8")
)

print("Audio files:", len(benchmark["audio_files"]))

audio_record = benchmark["audio_files"][0]

print(audio_record)

source_wav = Path(audio_record["wav_path"])

if (not source_wav.is_absolute()) :
    source_wav = PROJECT_ROOT / source_wav

print("Smoke source:", source_wav)
print("Exists:", source_wav.exists())

assert source_wav.exists()

Audio files: 10
{'video_id': 'L21_V015', 'wav_path': 'data/audio/L21_V015.wav', 'wav_sha256': '57fba586fca59952d745a03663a07f98960c7889e24760a928e62521f73c0d30', 'wav_size_bytes': 41727638, 'wav_sample_rate': 16000, 'wav_sample_count': 20863780, 'wav_meta_path': 'data/audio/L21_V015.wav.meta.json', 'wav_meta_sha256': 'ebcd153af8ba8778f336539cc248175a2feb5d3942c55a814e537846430e0a73'}
Smoke source: /content/drive/MyDrive/aic26/asr_model_comparison/data/audio/L21_V015.wav
Exists: True


## Run the new production pipeline on one video

In [7]:
from asr_retrieval.transcriber import transcribe_video

SMOKE_WORKSPACE = PROJECT_ROOT / "production_smoke" / "phase1"

artifact = transcribe_video(
    source_video = source_wav,
    workspace = SMOKE_WORKSPACE,
    config = config,
    transcriber = transcriber,
)

print("Video:", artifact.video.video_id)
print("Complete:", artifact.complete)
print("Windows:", len(artifact.windows))
print("Eligible:", sum(window.eligible for window in artifact.windows))

[NeMo I 2026-08-12 08:10:46 ctc_models:167] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[idx].timestep['word'/'segment'/'char']


[NeMo W 2026-08-12 08:10:46 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-12 08:10:46 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:02,  2.65s/it]

[NeMo I 2026-08-12 08:10:49 ctc_models:167] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[idx].timestep['word'/'segment'/'char']



[NeMo W 2026-08-12 08:10:49 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-12 08:10:49 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00,  2.77it/s]

[NeMo I 2026-08-12 08:10:50 ctc_models:167] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[idx].timestep['word'/'segment'/'char']


...

[NeMo I 2026-08-12 08:11:04 ctc_models:167] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[idx].timestep['word'/'segment'/'char']



[NeMo W 2026-08-12 08:11:04 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-12 08:11:04 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00,  2.76it/s]

[NeMo I 2026-08-12 08:11:05 ctc_models:167] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[idx].timestep['word'/'segment'/'char']



[NeMo W 2026-08-12 08:11:05 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-12 08:11:05 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00,  3.70it/s]


Video: L21_V015
Complete: True
Windows: 29
Eligible: 29


In [8]:
for window in artifact.windows[:5] :
    print("=" * 100)
    print(
        window.window.window_id,
        f"{window.window.start_s:.1f}s → {window.window.end_s:.1f}s",
    )
    print("status:", window.status)
    print("eligible:", window.eligible)
    print("warnings:", window.warning_reasons)
    print("rejections:", window.rejection_reasons)
    print("raw:")
    print(window.raw_text[:800])
    print()
    print("retrieval:")
    print(window.retrieval_text[:800])

transcript_path = (
    SMOKE_WORKSPACE
    / "transcripts"
    / f"{artifact.video.video_id}.json"
)

print(transcript_path)
print("Exists:", transcript_path.exists())
print("Size:", transcript_path.stat().st_size)

payload = json.loads(
    transcript_path.read_text(encoding = "utf-8")
)

print("Schema:", payload["schema_version"])
print("Complete:", payload["complete"])
print("Video:", payload["video"]["video_id"])
print("Source SHA:", payload["video"]["source_sha256"])
print("Windows:", len(payload["windows"]))
print("Parakeet:", payload["parakeet_identity"])
print("Postprocess:", payload["postprocess_version"])

L21_V015_0000 0.0s → 60.0s
status: ok
eligible: True
warnings: ()
rejections: ()
raw:
Ch kính chào quý vị. Đây là chương trình sáu mươi giây sáng của Đài Truyền hình Thành phố Hồ Chí Minh và chương trình hôm nay sẽ có những nội dung đáng chú ý sau đây Gió bỏ lệnh cấm xe ô tô cầu Rạch Miễu nhờ thử tải cầu kết thúc sớm hai ngày so với dự định. Nhà vệ sinh bệnh viện được nâng cấp đảm bảo sạch sẽ an toàn. Sa La la ốc đảo thu hút nhiều du khách đến Ô men. thưa quý vị lệnh cấm xe ô tô qua cầu Rạch Miễu sẽ được dỡ bỏ do đơn vị thi công thử tải hoàn thành sớm hơn dự kiến hai ngày. Thông tin vừa được bà Trần Thị Kim Uuyênên, Giám đốc Công ty trách nhiệm hữu hạn Bê  Ô tê C cầu Rạch Miu Bến Tre xác nhận.ư vậy, trong khung giờ hai mươi hai giờ đến bốn giờ sáng ngày mười bốn và mười lăm tháng tám, lệnh cấm xe ô tô qua cầu Rạch Miễu không còn hiệu lực.iệc lưu thông qua cầu sẽ diễn ra bìn

retrieval:
ch kính chào quý vị đây là chương trình sáu mươi giây sáng của đài truyền hình thành phố hồ chí minh 

## The critical resume smoke test

In [9]:
import time

started = time.perf_counter()

artifact_second = transcribe_video(
    source_video = source_wav,
    workspace = SMOKE_WORKSPACE,
    config = config,
    transcriber = transcriber,
)

elapsed = time.perf_counter() - started

print("Second run:", elapsed, "seconds")
print("Complete:", artifact_second.complete)
print("Windows:", len(artifact_second.windows))

Second run: 0.5539541979999285 seconds
Complete: True
Windows: 29


In [10]:
first_texts = [
    (
        window.window.window_id,
        window.raw_text,
        window.retrieval_text,
        window.eligible,
    )
    for window in artifact.windows
]

second_texts = [
    (
        window.window.window_id,
        window.raw_text,
        window.retrieval_text,
        window.eligible,
    )
    for window in artifact_second.windows
]

assert first_texts == second_texts

print("Resume output is exactly identical.")

Resume output is exactly identical.


In [11]:
transcriber.close()

print("Closed.")

Closed.
